In [5]:
from pathlib import Path
import os, sys

if Path.cwd().name == 'notebooks':
    os.chdir('..')
ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print('working from:', ROOT.name)

working from: project


In [6]:
# Imports & Utility Functions

import datetime as dt
import pandas as pd
import requests
from bs4 import BeautifulSoup
from dotenv import load_dotenv

# Ensure target raw data directory exists
RAW = Path('data/raw')
RAW.mkdir(parents=True, exist_ok=True)

# Reload environment variables from local secrets file
load_dotenv(override=True)
print('ALPHAVANTAGE_API_KEY loaded?', bool(os.getenv('ALPHAVANTAGE_API_KEY')))

def ts():
    """Generate a formatted timestamp string for filename versioning."""
    return dt.datetime.now().strftime('%Y%m%d-%H%M%S')

def save_csv(df: pd.DataFrame, prefix: str, **meta):
    """Export DataFrame to raw directory with structured metadata filename."""
    mid = '_'.join([f"{k}-{v}" for k, v in meta.items()])
    path = RAW / f"{prefix}_{mid}_{ts()}.csv"
    df.to_csv(path, index=False)
    print('Saved raw dataset to:', path)
    return path

def validate(df: pd.DataFrame, required):
    """Perform quality control and schema validation on DataFrames."""
    missing = [c for c in required if c not in df.columns]
    return {
        'missing_columns': missing,
        'shape': df.shape,
        'na_total': int(df.isna().sum().sum()),
        'dtypes': df.dtypes.to_dict()
    }

ALPHAVANTAGE_API_KEY loaded? True


In [9]:
# Data Source 1 — API Ingestion

SYMBOL = 'AAPL'
USE_ALPHA = bool(os.getenv('ALPHAVANTAGE_API_KEY')) and os.getenv('ALPHAVANTAGE_API_KEY') != 'your_api_key_here'

# Primary Route: Fetch daily price series via Alpha Vantage API
if USE_ALPHA:
    url = 'https://www.alphavantage.co/query'
    params = {'function': 'TIME_SERIES_DAILY', 'symbol': SYMBOL, 'outputsize': 'compact', 'apikey': os.getenv('ALPHAVANTAGE_API_KEY')}
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    js = r.json()
    key = [k for k in js if 'Time Series' in k]
    if not key:
        print('Alpha Vantage cap reached or error response:', str(list(js.values())[0])[:150])
        USE_ALPHA = False

if USE_ALPHA:
    df_api = pd.DataFrame(js[key[0]]).T.reset_index().rename(columns={'index': 'date', '4. close': 'close'})[['date', 'close']]
    df_api['date'] = pd.to_datetime(df_api['date'])
    df_api['close'] = pd.to_numeric(df_api['close'])

# Fallback Route: Download market series via yfinance if primary API fails
if not USE_ALPHA:
    import yfinance as yf
    print('Using yfinance fallback...')
    df_raw = yf.download(SYMBOL, period='3mo', interval='1d', auto_adjust=False)
    if isinstance(df_raw.columns, pd.MultiIndex):
        df_raw.columns = df_raw.columns.get_level_values(0)
    df_api = df_raw.reset_index()[['Date', 'Close']]
    df_api.columns = ['date', 'close']
    df_api['date'] = pd.to_datetime(df_api['date'])
    df_api['close'] = pd.to_numeric(df_api['close'])

# Validate schema and save raw dataset[cite: 1]
v_api = validate(df_api, ['date', 'close'])
print("API Data Validation Results:", v_api)
_ = save_csv(df_api.sort_values('date'), prefix='api', source='alpha' if USE_ALPHA else 'yfinance', symbol=SYMBOL)

API Data Validation Results: {'missing_columns': [], 'shape': (100, 2), 'na_total': 0, 'dtypes': {'date': dtype('<M8[ns]'), 'close': dtype('float64')}}
Saved raw dataset to: data\raw\api_source-alpha_symbol-AAPL_20260901-151142.csv


In [ ]:
# Data Source 2 — Web Scraping

SCRAPE_URL = 'https://en.wikipedia.org/wiki/Dow_Jones_Industrial_Average'
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}

try:
    resp = requests.get(SCRAPE_URL, headers=headers, timeout=30)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, 'html.parser')
    
    table = soup.find('table', {'class': 'wikitable'})
    rows = []
    for tr in table.find_all('tr'):
        cols = [c.get_text(strip=True) for c in tr.find_all(['th', 'td'])]
        if cols:
            rows.append(cols)
            
    header, *data = rows
    df_scrape = pd.DataFrame(data, columns=header)
except Exception as e:
    print('Scraping failed; switching to fallback table:', e)
    html = '<table><tr><th>Company</th><th>Symbol</th><th>Weight</th></tr><tr><td>Apple</td><td>AAPL</td><td>12.5</td></tr></table>'
    soup = BeautifulSoup(html, 'html.parser')
    rows = [[c.get_text(strip=True) for c in tr.find_all(['th', 'td'])] for tr in soup.find_all('tr')]
    header, *data = [r for r in rows if r]
    df_scrape = pd.DataFrame(data, columns=header)

# Standardize column headers and run schema checks
df_scrape.columns = [c.replace(' ', '_').lower() for c in df_scrape.columns]
v_scrape = validate(df_scrape, list(df_scrape.columns))
print("Scrape Validation Results:", v_scrape)

# Save raw output to data/raw/[cite: 1]
_ = save_csv(df_scrape, prefix='scrape', site='wikipedia', table='djia_components')

Scrape Validation Results: {'missing_columns': [], 'shape': (130, 4), 'na_total': 0, 'dtypes': {'year': dtype('O'), 'closingvalue': dtype('O'), 'netchange': dtype('O'), 'percentagechange': dtype('O')}}
Saved raw dataset to: data\raw\scrape_site-wikipedia_table-djia_components_20260901-145718.csv


## Stage 04: Data Ingestion Summary

### Sources & Parameters
- **Market Data API**: Alpha Vantage (`TIME_SERIES_DAILY`) targeting `AAPL` with defensive fallback logic using `yfinance`.
- **Scraped Data**: Public Wikipedia table covering DJIA component stocks parsed via BeautifulSoup.

### Validation & Schema Control
- Verified key columns (`date`, `close`) exist and contain valid types (`datetime64`, `float64`).
- Assessed missingness (`na_total`) and dimensions (`shape`) before saving.

### Governance & Security
- Confirmed `.env` secrets file remains untracked in `.gitignore`.
- Raw output is timestamped and written to `data/raw/` to ensure pipeline reproducibility.

In [11]:
# ==========================================
# Stage 05: Data Storage Layer Implementation
# ==========================================
import os, pathlib, typing as t
import pandas as pd
from dotenv import load_dotenv

# 1. Environment-driven path resolution via os.getenv
load_dotenv(override=True)
RAW_DIR = pathlib.Path(os.getenv('DATA_DIR_RAW', 'data/raw'))
PROC_DIR = pathlib.Path(os.getenv('DATA_DIR_PROCESSED', 'data/processed'))

# Automatically guarantee existence of target data directories
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROC_DIR.mkdir(parents=True, exist_ok=True)

# 2. Reusable I/O utility functions
def detect_format(path: t.Union[str, pathlib.Path]) -> str:
    """Determine file storage format based on extension suffix."""
    s = str(path).lower()
    if s.endswith('.csv'): 
        return 'csv'
    if s.endswith(('.parquet', '.pq', '.parq')): 
        return 'parquet'
    raise ValueError(f"Unsupported format extension: {s}")

def write_df(df: pd.DataFrame, path: t.Union[str, pathlib.Path]) -> pathlib.Path:
    """Write DataFrame with parent directory creation and format routing."""
    p = pathlib.Path(path)
    p.parent.mkdir(parents=True, exist_ok=True)
    fmt = detect_format(p)
    if fmt == 'csv':
        df.to_csv(p, index=False)
    else:
        try:
            df.to_parquet(p, index=False)
        except Exception as e:
            raise RuntimeError("Parquet engine missing. Install pyarrow or fastparquet.") from e
    return p

def read_df(path: t.Union[str, pathlib.Path]) -> pd.DataFrame:
    """Read dataset dynamically with format detection and date parsing."""
    p = pathlib.Path(path)
    if not p.exists():
        raise FileNotFoundError(f"Target dataset path does not exist: {p}")
    fmt = detect_format(p)
    if fmt == 'csv':
        header = pd.read_csv(p, nrows=0).columns
        return pd.read_csv(p, parse_dates=['date']) if 'date' in header else pd.read_csv(p)
    else:
        return pd.read_parquet(p)

# 3. Store raw landing dataset and processed binary dataset safely
raw_csv_path = None
proc_pq_path = None

if 'df_api' in locals() and df_api is not None:
    raw_csv_path = RAW_DIR / f"market_data_{ts()}.csv"
    proc_pq_path = PROC_DIR / f"market_data_{ts()}.parquet"
    
    write_df(df_api, raw_csv_path)
    print(f"Raw dataset written to: {raw_csv_path}")
    
    try:
        write_df(df_api, proc_pq_path)
        print(f"Processed dataset written to: {proc_pq_path}")
    except RuntimeError as err:
        print(f"Parquet engine notice: {err}")
else:
    print("[WARNING] 'df_api' is not found in memory. Please execute Cell 3 first.")

# 4. Storage reload verification check
if raw_csv_path and raw_csv_path.exists():
    reloaded_csv = read_df(raw_csv_path)
    assert reloaded_csv.shape == df_api.shape, "Shape mismatch detected in reloaded data"
    print("Storage Validation Check: Shape and schema successfully verified.")

Raw dataset written to: data\raw\market_data_20260901-151227.csv
Processed dataset written to: data\processed\market_data_20260901-151227.parquet
Storage Validation Check: Shape and schema successfully verified.
